# Setup — Reward Model, SFT, Preference Data

Run this notebook **once**. Everything is saved to Google Drive so the training
notebook (`02_train.ipynb`) can be run separately for each method.

**Steps:**
1. Mount Drive and install dependencies
2. Train the oracle reward model (GPT-2 Large classifier)
3. Train the SFT reference model (GPT-2 Large, positive reviews)
4. Generate 500 preference pairs

In [ ]:
# Mount Drive — all outputs persist here across sessions
from google.colab import drive
drive.mount('/content/drive')

!pip install -q 'transformers>=4.40' datasets 'trl>=0.8' matplotlib

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, GPT2LMHeadModel, GPT2ForSequenceClassification,
    TrainingArguments, Trainer,
)
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

# ── all outputs go to Drive so they survive session restarts ─────────────────
ROOT       = '/content/drive/MyDrive/cal-simpo-outputs'
REWARD_DIR = f'{ROOT}/reward_model'
SFT_DIR    = f'{ROOT}/sft_model'
PREF_DIR   = f'{ROOT}/pref_data'
for d in [REWARD_DIR, SFT_DIR, PREF_DIR]:
    os.makedirs(d, exist_ok=True)

BASE   = 'gpt2-large'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

PROMPT_LEN = 32
MAX_NEW    = 128
NUM_PREF   = 500
SEED       = 42

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token    = tok.eos_token
tok.padding_side = 'right'
print(f'Device: {DEVICE}  |  model: {BASE}  |  root: {ROOT}')

## Step 1 — Oracle Reward Model

In [ ]:
imdb     = load_dataset('imdb')
rm_train = imdb['train'].shuffle(seed=SEED).select(range(5000))
rm_val   = imdb['test'].shuffle(seed=SEED).select(range(1000))

def tok_cls(batch):
    return tok(batch['text'], truncation=True, max_length=512, padding='max_length')

rm_train = (rm_train.map(tok_cls, batched=True, remove_columns=['text'])
                    .rename_column('label', 'labels'))
rm_val   = (rm_val.map(tok_cls,   batched=True, remove_columns=['text'])
                   .rename_column('label', 'labels'))
rm_train.set_format('torch');  rm_val.set_format('torch')

reward_model = GPT2ForSequenceClassification.from_pretrained(BASE, num_labels=2)
reward_model.config.pad_token_id = tok.pad_token_id

rm_trainer = Trainer(
    model=reward_model,
    args=TrainingArguments(
        output_dir=REWARD_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        fp16=(DEVICE == 'cuda'),
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True,
        seed=SEED, logging_steps=100, report_to='none',
    ),
    train_dataset=rm_train,
    eval_dataset=rm_val,
)
rm_trainer.train()
reward_model = rm_trainer.model
reward_model.save_pretrained(REWARD_DIR)
tok.save_pretrained(REWARD_DIR)
print(f'Reward model saved to {REWARD_DIR}')

In [ ]:
# Helper used in Step 3 to score completions
@torch.no_grad()
def oracle_reward(texts, batch_size=16):
    """Returns (N,) log-odds rewards: log p(pos) - log p(neg)."""
    reward_model.eval().to(DEVICE)
    parts = []
    for i in range(0, len(texts), batch_size):
        enc    = tok(texts[i:i+batch_size], truncation=True, max_length=512,
                     padding=True, return_tensors='pt').to(DEVICE)
        lp     = F.log_softmax(reward_model(**enc).logits, dim=-1)
        parts.append((lp[:, 1] - lp[:, 0]).cpu())
    return torch.cat(parts)

## Step 2 — SFT Reference Model

In [ ]:
pos    = imdb['train'].filter(lambda x: x['label'] == 1).shuffle(seed=SEED).select(range(5000))
pos_ds = Dataset.from_dict({'text': pos['text']})

sft_trainer = SFTTrainer(
    model=BASE,
    processing_class=tok,
    args=SFTConfig(
        output_dir=SFT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        max_length=256,              # max_seq_length renamed to max_length in trl >= 0.8
        dataset_text_field='text',
        fp16=(DEVICE == 'cuda'),
        seed=SEED, save_strategy='epoch',
        logging_steps=50, report_to='none',
    ),
    train_dataset=pos_ds,
)
sft_trainer.train()
sft_trainer.save_model(SFT_DIR)
tok.save_pretrained(SFT_DIR)
del sft_trainer;  torch.cuda.empty_cache()
print(f'SFT model saved to {SFT_DIR}')

## Step 3 — Preference Pairs

In [ ]:
sft = GPT2LMHeadModel.from_pretrained(SFT_DIR).to(DEVICE)
sft.eval()

test_subset = imdb['test'].shuffle(seed=SEED).select(range(NUM_PREF))
pairs = []

for ex in tqdm(test_subset, desc='Building preference pairs'):
    ids    = tok.encode(ex['text'], add_special_tokens=False)[:PROMPT_LEN]
    prompt = tok.decode(ids, skip_special_tokens=True)
    inp    = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = sft.generate(
            inp, max_new_tokens=MAX_NEW, do_sample=True, temperature=1.0,
            num_return_sequences=2, pad_token_id=tok.eos_token_id,
        )
    comps = [tok.decode(out[k][len(ids):], skip_special_tokens=True) for k in range(2)]

    r    = oracle_reward([prompt + comps[0], prompt + comps[1]])
    w, l = (0, 1) if r[0] >= r[1] else (1, 0)
    pairs.append({
        'idx': len(pairs), 'prompt': prompt,
        'chosen': comps[w],   'rejected': comps[l],
        'chosen_reward': r[w].item(), 'rejected_reward': r[l].item(),
    })

del sft;  torch.cuda.empty_cache()
reward_model.cpu();  torch.cuda.empty_cache()

rng  = np.random.default_rng(SEED)
shuf = rng.permutation(len(pairs)).tolist()
n    = int(len(pairs) * 0.9)
json.dump([pairs[i] for i in shuf[:n]], open(f'{PREF_DIR}/train.json', 'w'))
json.dump([pairs[i] for i in shuf[n:]], open(f'{PREF_DIR}/val.json',   'w'))
print(f'Saved {n} train + {len(pairs)-n} val pairs to {PREF_DIR}')
print('Setup complete. Run 02_train.ipynb next (set METHOD at the top of that notebook).')